# 03 — Modelagem e Avaliação

Agora vem a parte mais empolgante: treinar e comparar os modelos!

Vamos testar 4 abordagens diferentes e ver qual performa melhor:
1. **Regressão Linear** — o modelo mais simples, serve como baseline
2. **Ridge** — regressão linear com regularização
3. **Random Forest** — conjunto de árvores de decisão
4. **XGBoost** — gradient boosting, geralmente o melhor

Métricas usadas:
- **RMSE**: erro quadrático médio — penaliza erros grandes
- **R²**: % da variância explicada pelo modelo (1.0 = perfeito)

## 1. Imports e Preparação

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.join(os.getcwd(), '..'))
from src.preprocessor import prepare_data
from src.model import get_models, evaluate_with_cv, compare_models, train_best_model, plot_feature_importance, plot_predictions_vs_actual
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Carrega e prepara os dados
df = pd.read_csv('../data/train.csv')
X, y = prepare_data(df)

# Split treino/teste — 80% treino, 20% teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')

## 2. Comparação de Modelos com Cross-Validation

Usamos 5-fold cross-validation para ter uma estimativa mais confiável da performance. Isso evita que o modelo pareça bom só porque teve sorte com o split de treino/teste.

In [ ]:
print('Comparando modelos... isso pode levar alguns segundos!')
df_resultados = compare_models(X_train.values, y_train.values)

print('\n📋 Ranking de Modelos (menor RMSE = melhor):')
df_resultados

## 3. Visualização dos Resultados

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cores = ['#55A868' if i == 0 else '#4C72B0' for i in range(len(df_resultados))]
ax.barh(df_resultados['Modelo'], df_resultados['RMSE_mean'], color=cores)
ax.set_xlabel('RMSE (menor = melhor)')
ax.set_title('Comparação de Modelos — Cross-Validation RMSE')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'🏆 Melhor modelo: {df_resultados.iloc[0]["Modelo"]}')

## 4. Treinamento Final e Métricas no Teste

In [ ]:
os.makedirs('../outputs', exist_ok=True)

resultado = train_best_model(
    X_train.values, y_train.values,
    X_test.values, y_test.values
)

## 5. Importância das Features

Quais características do imóvel mais influenciam o preço?

In [ ]:
plot_feature_importance(
    resultado['model'],
    feature_names=list(X.columns),
    save_path='../outputs/feature_importance.png'
)

## 6. Predições vs Valores Reais

Se o modelo fosse perfeito, todos os pontos estariam na linha vermelha diagonal.

In [ ]:
preds_log = resultado['model'].predict(X_test.values)

plot_predictions_vs_actual(
    y_true=np.expm1(y_test.values),
    y_pred=np.expm1(preds_log),
    save_path='../outputs/predictions_vs_actual.png'
)

print(f"\n🎯 Resumo final:")
print(f"   Modelo: {resultado['name']}")
print(f"   RMSE:   $ {resultado['RMSE']:,.0f}")
print(f"   MAE:    $ {resultado['MAE']:,.0f}")
print(f"   R²:     {resultado['R2']:.4f}")